In [52]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm


In [53]:

# Make torch deterministic
_ = torch.manual_seed(0)



In [54]:

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

# Load the MNIST dataset
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# Create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load the MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Define the device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [55]:
# Create an overly expensive neural network to classify MNIST digits
# Daddy got money, so I don't care about efficiency
class RichBoyNet(nn.Module):
    def __init__(self, hidden_size_1=1000, hidden_size_2=2000):
        super(RichBoyNet,self) .__init__()
        self.linear1 = nn.Linear(28*28, hidden_size_1)
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()
        
    def forward(self, img):
        x = img.view(-1, 28*28)
        x = self.relu(self.linear1(x))
        x= self.relu(self.linear2(x))
        x = self.linear3(x)
        return x
    
net = RichBoyNet().to(device)
    
    

In [56]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    
    total_iterations = 0
    
    for epoch in range(epochs):
        net.train()
        
        loss_sum = 0
        num_iterations = 0
        
        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            output = net(x.view(-1, 28*28))
            loss = cross_el(output, y)
            loss_sum += loss.item()
            avg_loss = loss_sum / num_iterations
            data_iterator.set_postfix(loss=avg_loss)
            loss.backward()
            optimizer.step()
            
            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

train(train_loader, net, epochs=1)

Epoch 1: 100%|██████████| 6000/6000 [00:40<00:00, 148.76it/s, loss=0.238]


In [57]:
original_weights = {}
for name, param in net.named_parameters():
        original_weights[name] = param.clone().detach()

In [58]:
def test():
    correct = 0
    total = 0
    
    wrong_counts = [0 for i in range(10)]
    
    with torch.no_grad():
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = net(x.view(-1, 784))
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct +=1
                else:
                    wrong_counts[y[idx]] +=1
                total +=1
    print(f'Accuracy: {round(correct/total, 3)}')
    for i in range(len(wrong_counts)):
        print(f'wrong counts for the digit {i}: {wrong_counts[i]}')

test()

Testing: 100%|██████████| 1000/1000 [00:05<00:00, 196.63it/s]

Accuracy: 0.954
wrong counts for the digit 0: 12
wrong counts for the digit 1: 19
wrong counts for the digit 2: 36
wrong counts for the digit 3: 92
wrong counts for the digit 4: 29
wrong counts for the digit 5: 41
wrong counts for the digit 6: 38
wrong counts for the digit 7: 47
wrong counts for the digit 8: 35
wrong counts for the digit 9: 115


In [59]:
# Print the size of the weights matrices of the network
# Save the countrof the total number of parameters
total_parameters_original = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    total_parameters_original += layer.weight.nelement() + layer.bias.nelement()
    print(f'Layer {index+1}: W: {layer.weight.shape} + B: {layer.bias.shape}')
print(f'Total number of parameters: {total_parameters_original:,}')

Layer 1: W: torch.Size([1000, 784]) + B: torch.Size([1000])
Layer 2: W: torch.Size([2000, 1000]) + B: torch.Size([2000])
Layer 3: W: torch.Size([10, 2000]) + B: torch.Size([10])
Total number of parameters: 2,807,010


In [60]:
class LoRAParametrization(nn.Module):
    def __init__(self, features_in, features_out, rank=1, alpha=1, device='cpu'):
        super() .__init__()
        # Section 4.1 of the paper:
        # We use a random Gaussian initialization for A and zero for B, so AW = BA is zero at the beginning of training
        self.lora_A = nn.Parameter(torch.zeros((rank, features_out)).to(device))
        self.lora_B = nn.Parameter(torch.zeros((features_in, rank)).to(device))
        nn.init.normal_(self.lora_A, mean=0, std=1)
        
        # Section 4.1 of the paper:
        # We then scale AWx by a/r , where a is a constant in r.
        # When optimizing with Adam, tuning a is roughly the same as tuning the learning rate if we scale the initialization appropriat
        # As a result, we simply set a to the first r we try and do not tune it.
        # This scaling helps to reduce the need to retune hyperparameters when we vary r.
        self.scale = alpha / rank
        self.enabled = True
        
    def forward(self, original_weights):
        if self.enabled:
            # Return X + (B*A)*scale
            return original_weights + torch.matmul(self.lora_B, self.lora_A).view(original_weights.shape) * self.scale
        else:
            return original_weights


In [61]:
import torch.nn.utils.parametrize as parametrize

def linear_layer_parameterization(layer, device, rank=1, lora_alpha=1):
    features_in, features_out = layer.weight.shape
    return LoRAParametrization(features_in, features_out, rank=rank, alpha=lora_alpha, device=device)

# ALL THREE lines now have the correct closing parentheses
parametrize.register_parametrization(net.linear1, "weight", linear_layer_parameterization(net.linear1, device))
parametrize.register_parametrization(net.linear2, "weight", linear_layer_parameterization(net.linear2, device))
parametrize.register_parametrization(net.linear3, "weight", linear_layer_parameterization(net.linear3, device))

def enable_disable_lora(enabled=True):
    for layer in [net.linear1, net.linear2, net.linear3]:
        layer.parametrizations["weight"][0].enabled = enabled

In [62]:
total_parameters_lora = 0
total_parameters_non_lora = 0
for index, layer in enumerate([net.linear1, net.linear2, net.linear3]):
    total_parameters_lora += layer.parametrizations["weight"][0].lora_A.nelement() + layer.parametrizations["weight"][0].lora_B.nelement()
    total_parameters_non_lora += layer.weight.nelement() + layer.bias.nelement()
    print(f'Layer {index+1}: w: {layer.weight. shape} + B: {layer.bias. shape} + lora_A: {layer.parametrizations["weight"][0].lora_A. shape} + lora_B: {layer.parametrizations}')

# The non-LoRA parameters count must match the original network
assert total_parameters_non_lora == total_parameters_original
print(f'Total number of parameters (original): {total_parameters_non_lora:,}')
print(f'Total number of parameters (original + LoRA): {total_parameters_lora + total_parameters_non_lora:,}')
print(f'Parameters introduced by LoRA: {total_parameters_lora:,}')
parameters_incremment = (total_parameters_lora / total_parameters_non_lora) * 100
print(f'Parameters increment: {parameters_incremment:.3f}%')

Layer 1: w: torch.Size([1000, 784]) + B: torch.Size([1000]) + lora_A: torch.Size([1, 784]) + lora_B: ModuleDict(
  (weight): ParametrizationList(
    (0): LoRAParametrization()
  )
)
Layer 2: w: torch.Size([2000, 1000]) + B: torch.Size([2000]) + lora_A: torch.Size([1, 1000]) + lora_B: ModuleDict(
  (weight): ParametrizationList(
    (0): LoRAParametrization()
  )
)
Layer 3: w: torch.Size([10, 2000]) + B: torch.Size([10]) + lora_A: torch.Size([1, 2000]) + lora_B: ModuleDict(
  (weight): ParametrizationList(
    (0): LoRAParametrization()
  )
)
Total number of parameters (original): 2,807,010
Total number of parameters (original + LoRA): 2,813,804
Parameters introduced by LoRA: 6,794
Parameters increment: 0.242%


In [63]:
# Freeze the non-Lora parameters
for name, param in net.named_parameters():
    if 'lora' not in name:
        print(f'Freezing non-LoRA parameter {name}')
        param.requires_grad = False

# Load the MNIST dataset again, by keeping only the digit 9
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
exclude_indices = mnist_trainset. targets == 9
mnist_trainset.data = mnist_trainset.data[exclude_indices]
mnist_trainset.targets = mnist_trainset.targets[exclude_indices]
# Create a dataloader for the training
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Train the network with LoRA only on the digit 9 and only for 100 bacbes (hoping that it would improve the performance on the digit 9)
train(train_loader, net, epochs=1, total_iterations_limit=100)

Freezing non-LoRA parameter linear1.bias
Freezing non-LoRA parameter linear1.parametrizations.weight.original
Freezing non-LoRA parameter linear2.bias
Freezing non-LoRA parameter linear2.parametrizations.weight.original
Freezing non-LoRA parameter linear3.bias
Freezing non-LoRA parameter linear3.parametrizations.weight.original


Epoch 1:  99%|█████████▉| 99/100 [00:00<00:00, 102.41it/s, loss=0.145]


In [65]:
# Check that the frozen parameters are still unchanged by the finetuning
assert torch.all(net.linear1.parametrizations.weight.original == original_weights['linear1.weight'])
assert torch.all(net.linear2.parametrizations.weight.original == original_weights['linear2.weight'])
assert torch.all(net.linear3.parametrizations.weight.original == original_weights['linear3.weight'])

enable_disable_lora(enabled=True)
# The new linear1.weight is obtained by the "forward" function of our LoRA parametrization
# The original weights have been moved to net.linear1.parametrizations.weight.original
# More info here: https://pytorch.org/tutorials/intermediate/parametrizations.htmlwinspecting-a-paranetrized-module
lora_layer = net.linear1.parametrizations.weight[0]
manual_weight = net.linear1.parametrizations.weight.original + (lora_layer.lora_B @ lora_layer.lora_A) * lora_layer.scale
assert torch.allclose(net.linear1.weight, manual_weight)

enable_disable_lora(enabled=False)
#If we disable LoRA, the linear1.weight is the original one
assert torch.equal(net.linear1.weight, original_weights['linear1.weight'])

In [66]:
# Test with LoRA enabled
enable_disable_lora(enabled=True)
test()

Testing: 100%|██████████| 1000/1000 [00:06<00:00, 160.38it/s]

Accuracy: 0.804
wrong counts for the digit 0: 20
wrong counts for the digit 1: 23
wrong counts for the digit 2: 83
wrong counts for the digit 3: 401
wrong counts for the digit 4: 388
wrong counts for the digit 5: 257
wrong counts for the digit 6: 81
wrong counts for the digit 7: 310
wrong counts for the digit 8: 387
wrong counts for the digit 9: 7


In [67]:
# Test with LoRA disabled

enable_disable_lora(enabled=False)
test()

Testing: 100%|██████████| 1000/1000 [00:07<00:00, 135.44it/s]

Accuracy: 0.954
wrong counts for the digit 0: 12
wrong counts for the digit 1: 19
wrong counts for the digit 2: 36
wrong counts for the digit 3: 92
wrong counts for the digit 4: 29
wrong counts for the digit 5: 41
wrong counts for the digit 6: 38
wrong counts for the digit 7: 47
wrong counts for the digit 8: 35
wrong counts for the digit 9: 115
